# [실습] 네이버 영화 리뷰 감성 분류 Fine-Tuning

앞서 뉴스 주제 분류(KLUE-YNAT, 7개 클래스)를 해봤습니다.  
이번에는 **네이버 영화 리뷰 감성 분류(NSMC, 2개 클래스)** 를 직접 구현해봅니다.

| 항목 | 데모에서 한 것 | 이번 실습 |
|------|----------------|----------|
| 데이터셋 | `klue/ynat` (뉴스 주제) | `nsmc` (영화 리뷰 감성) |
| 모델 | `klue/bert-base` | `klue/bert-base` (동일) |
| 분류 수 | 7개 (IT과학, 경제, …) | 2개 (긍정/부정) |
| 텍스트 컬럼 | `title` | `document` |
| 레이블 컬럼 | `label` (0~6) | `label` (0 또는 1) |

> **핵심 포인트**: 코드 구조는 데모와 거의 동일합니다.  
> "어디를 바꿔야 하는지" 찾는 것이 이번 실습의 목표입니다.

---
## 1단계: 패키지 설치

필요한 패키지를 설치하세요.

- `transformers` — 모델과 토크나이저
- `datasets` — HuggingFace 데이터셋 로드
- `accelerate` — 학습 가속
- `scikit-learn` — 평가 지표 계산

데모 노트북에서 설치했던 것과 동일합니다.  
`torchvision` 충돌 방지를 위해 먼저 제거하는 것도 잊지 마세요.

---
## 2단계: 라이브러리 임포트 및 GPU 확인

아래 라이브러리들을 임포트하세요:

- `torch` — GPU 사용 여부 확인
- `numpy` — 배열 연산
- `datasets`의 `load_dataset` — 데이터셋 불러오기
- `transformers`에서:
  - `AutoTokenizer` — 토크나이저
  - `AutoModelForSequenceClassification` — 분류 모델
  - `TrainingArguments` — 학습 설정
  - `Trainer` — 학습 실행
  - `EarlyStoppingCallback` — 조기 종료
- `sklearn.metrics`에서 `accuracy_score`, `f1_score`, `classification_report`, `confusion_matrix`
- `matplotlib.pyplot`, `seaborn` — 시각화

임포트 후, `torch.device`로 GPU가 사용 가능한지 확인하고 출력하세요.

---
## 3단계: 데이터셋 로드 및 탐색

HuggingFace에서 **NSMC** 데이터셋을 불러오세요.

```python
load_dataset("nsmc")
```

불러온 뒤 다음을 확인하세요:
- 전체 구조 (`print(dataset)`) — train/test로 나뉘어 있습니다
- 컬럼 이름 확인 — `id`, `document`, `label` 3개 컬럼이 있습니다
- 샘플 데이터 3개 정도 출력
- `label`의 분포 확인 (0과 1이 각각 몇 개인지)

> **참고**: NSMC의 레이블은 `0 = 부정`, `1 = 긍정`입니다.

---
## 4단계: 레이블 매핑 및 데이터 분할

### 레이블 매핑
NSMC는 이진 분류이므로:
- `num_labels = 2`
- `id2label = {0: "부정", 1: "긍정"}`
- `label2id = {"부정": 0, "긍정": 1}`

### 데이터 분할
NSMC는 `train`과 `test`만 제공합니다 (validation이 없습니다).  
따라서 `train`을 **train/validation으로 분할**해야 합니다.

- `dataset["train"].train_test_split()`을 사용하세요
- `test_size=0.1` 정도로 validation 비율을 설정
- `seed=42`로 재현성 확보
- `stratify_by_column="label"`로 레이블 비율 유지

최종적으로 `train_dataset`, `val_dataset`, `test_dataset` 3개를 만들고, 각각의 크기를 출력하세요.

> **주의**: NSMC의 `document` 컬럼에는 간혹 `None` 값이 있습니다.  
> 분할 전에 `dataset.filter(lambda x: x["document"] is not None)`으로 제거하세요.

---
## 5단계: 토크나이저 로드 및 토큰화

### 토크나이저 로드
모델 이름은 데모와 동일하게 `"klue/bert-base"`를 사용합니다.  
`AutoTokenizer.from_pretrained()`로 불러오세요.

### 토큰화 함수 작성
토큰화 함수를 만드세요. 핵심 설정:
- **텍스트 컬럼**: NSMC는 `"document"` 컬럼에 텍스트가 있습니다 (데모의 `"title"`과 다릅니다!)
- `padding="max_length"` — 고정 길이로 패딩
- `truncation=True` — 긴 텍스트 잘라내기
- `max_length=128`

### 토큰화 적용
- `train_dataset`, `val_dataset`, `test_dataset` 모두에 `.map(tokenize_fn, batched=True)` 적용
- `.set_format("torch", columns=["input_ids", "attention_mask", "label"])`로 PyTorch 형식 설정
- 샘플 토큰을 디코딩해서 잘 변환되었는지 확인

---
## 6단계: 모델 로드

`AutoModelForSequenceClassification.from_pretrained()`로 모델을 불러오세요.

설정할 인자:
- `model_name`: `"klue/bert-base"`
- `num_labels`: 이진 분류이므로 2
- `id2label`, `label2id`: 4단계에서 만든 매핑

불러온 뒤:
- `.to(device)`로 GPU에 올리세요
- 전체 파라미터 수를 출력하세요

---
## 7단계: 학습 설정 및 Fine-Tuning

### 평가 함수
`compute_metrics` 함수를 작성하세요:
- 입력: `eval_pred` (logits, labels 튜플)
- logits에서 `np.argmax`로 예측값 추출
- `accuracy_score`와 `f1_score(average="weighted")`를 계산해서 딕셔너리로 반환

### TrainingArguments 설정
데모와 동일한 설정을 사용하되, 아래 사항을 참고하세요:

| 설정 | 값 | 설명 |
|------|----|------|
| `output_dir` | `"./nsmc_results"` | 결과 저장 경로 |
| `num_train_epochs` | 3 | |
| `per_device_train_batch_size` | 32 | |
| `per_device_eval_batch_size` | 64 | |
| `learning_rate` | 2e-5 | |
| `weight_decay` | 0.01 | |
| `warmup_ratio` | 0.1 | |
| `eval_strategy` | `"epoch"` | 매 에폭마다 평가 |
| `save_strategy` | `"epoch"` | 매 에폭마다 저장 |
| `load_best_model_at_end` | `True` | |
| `metric_for_best_model` | `"f1"` | |
| `fp16` | GPU 사용 시 True | |
| `report_to` | `"none"` | wandb 등 비활성화 |

### Trainer 생성 및 학습
- `Trainer`에 모델, 학습 인자, train/eval 데이터셋, 평가 함수, EarlyStoppingCallback을 넣으세요
- `trainer.train()`으로 학습을 실행하세요

> **참고**: NSMC train은 약 15만 건으로, Colab T4에서 에폭당 약 5~10분 걸립니다.  
> 시간이 부족하면 `num_train_epochs=1`로 줄이거나, train 데이터를 `.select(range(30000))` 등으로 축소해보세요.

---
## 8단계: 학습 곡선 시각화

`trainer.state.log_history`에서 학습 로그를 꺼내서 그래프를 그려보세요.

그려야 할 그래프 3개:
1. **Training Loss** — step별 학습 손실 (x: step, y: loss)
2. **Validation Loss** — epoch별 검증 손실 (x: epoch, y: eval_loss)
3. **Validation Metrics** — epoch별 accuracy와 f1 (x: epoch, y: score)

`plt.subplots(1, 3, figsize=(18, 5))`로 한 줄에 3개 그래프를 배치하면 보기 좋습니다.

> **힌트**: `log_history`는 딕셔너리 리스트입니다.  
> `"loss"` 키가 있으면 학습 로그, `"eval_loss"` 키가 있으면 평가 로그입니다.

---
## 9단계: 테스트셋 평가

`trainer.evaluate(test_dataset)`으로 테스트셋 성능을 측정하세요.

출력할 지표:
- Loss
- Accuracy
- F1 Score

> **기대 성능**: BERT-base로 NSMC를 학습하면 보통 **accuracy 88~90%** 정도 나옵니다.

---
## 10단계: Classification Report & Confusion Matrix

### Classification Report
- `trainer.predict(test_dataset)`으로 전체 예측 수행
- `np.argmax(predictions.predictions, axis=-1)`로 예측 레이블 추출
- `classification_report()`로 precision, recall, f1을 클래스별로 출력
- `target_names`에 `["부정", "긍정"]`을 넣으세요

### Confusion Matrix
- `confusion_matrix()`로 혼동 행렬 계산
- `seaborn.heatmap()`으로 시각화 (annot=True, fmt="d", cmap="Blues")
- x축/y축 레이블에 "부정", "긍정"을 넣으세요

---
## 11단계: 추론 테스트

학습된 모델로 직접 영화 리뷰 문장을 넣어서 감성을 예측해보세요.

### 추론 함수 작성
텍스트를 받아서 긍정/부정과 확률을 반환하는 함수를 만드세요:
1. `tokenizer()`로 텍스트를 토큰화 (`return_tensors="pt"`, `truncation=True`, `padding=True`)
2. 토큰을 GPU로 이동
3. `model.eval()` + `torch.no_grad()` 안에서 `model(**inputs)` 실행
4. `torch.softmax()`로 확률 변환
5. `argmax`로 예측 클래스, 해당 확률을 반환

### 테스트 문장 예시
아래 리뷰들을 넣어서 결과를 확인해보세요 (물론 직접 추가해도 됩니다!):

```python
test_reviews = [
    "연기도 좋고 스토리도 탄탄해서 시간 가는 줄 몰랐어요",
    "돈 아까운 영화... 다시는 안 볼 듯",
    "그냥 그저 그런 영화",
    "배우들 연기력이 미쳤다 진짜 몰입감 장난 아님",
    "스토리가 너무 뻔하고 지루했음",
]
```

---
## 도전 과제 (선택)

시간이 남으면 아래를 시도해보세요:

1. **하이퍼파라미터 실험**: `learning_rate`를 `5e-5`, `1e-5`로 바꿔보고 성능 차이를 비교해보세요
2. **다른 모델 사용**: `"klue/bert-base"` 대신 `"monologg/koelectra-base-v3-discriminator"`를 사용해보세요 — 코드에서 모델 이름만 바꾸면 됩니다
3. **오분류 분석**: 모델이 틀린 샘플들을 뽑아서 "왜 틀렸을까?" 살펴보세요